In [ ]:
# ✅ STEP 1: Install dependencies
!pip install transformers shap --quiet

In [ ]:
# ✅ STEP 2: Import necessary libraries
import shap
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline, BertTokenizer, BertForSequenceClassification
from google.colab import files
import zipfile
import os

In [ ]:
# ✅ STEP 3: Upload model zip and test.csv
print("Please upload your fine-tuned model .zip file:")
uploaded = files.upload()  # Upload e.g., bert_sentiment_model.zip


In [ ]:
# ✅ STEP 4: Extract model
model_zip_filename = next(iter(uploaded))
model_extract_path = "/content/bert_sentiment_model"

with zipfile.ZipFile(model_zip_filename, "r") as zip_ref:
    zip_ref.extractall(model_extract_path)

print("Model extracted to:", model_extract_path)


In [ ]:
# ✅ STEP 5: Load the model
tokenizer = BertTokenizer.from_pretrained(model_extract_path)
model = BertForSequenceClassification.from_pretrained(model_extract_path)
model.eval()

In [ ]:
# ✅ STEP 6: Create HuggingFace pipeline
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=True)


In [ ]:
# ✅ STEP 7: Upload test.csv
print("Please upload your test.csv file:")
uploaded = files.upload()  # Upload test.csv
test_csv_filename = next(iter(uploaded))


In [ ]:
# ✅ STEP 8: Load test data
df = pd.read_csv(test_csv_filename)
texts = df["text"].astype(str).tolist()
sample_texts = texts[:50]  # Limit to 25 examples for SHAP

In [ ]:
# ✅ STEP 9: Run SHAP explainer
explainer = shap.Explainer(pipe)
shap_values = explainer(sample_texts)


In [ ]:
# ✅ STEP 10: Display all 25 SHAP explanations cleanly
for i in range(len(sample_texts)):
    print(f"\n--- SHAP Explanation for Example {i+1} ---")
    shap.plots.text(shap_values[i])
